**1. Import different config files** 
<br> <br>
We will start the notebook with OmegaConfig, which exists to combine multiple configuration files into one tree. This optimizes workflow and reduces the need for a crowded hierarchy of config files. Features include adding to the tree with dotlist notation, merging with overrides, and checking variables against a typed schema.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from omegaconf import OmegaConf

base_dir = Path.cwd()

market = OmegaConf.load(str(base_dir / "conf" / "market" / "aapl.yaml"))              # stock data
strategy = OmegaConf.load(str(base_dir / "conf" / "strategy" / "sma_crossover.yaml"))  # details in engine.py
backtest = OmegaConf.load(str(base_dir / "conf" / "backtest" / "default.yaml"))        # initial stock account parameters

# tree made from market.yaml, strategy.yaml, backtest.yaml
cfg = OmegaConf.create({"market": market, "strategy": strategy, "backtest": backtest})
print(OmegaConf.to_yaml(cfg))
print(cfg.market.ticker, cfg.strategy.fast_window)

: 

**2. Merging with override**
<br> <br>
Instead of manually editing the file architecture within VS code, you can add additional parameters to your config files straight from the notebook. It also saves the need for the "get" function when pulling the hyperparameter values into your model code.

In [ ]:
# formatting: config_name.param = value
dotlist = ["strategy.fast_window=5", "backtest.commission=0.002"]
override = OmegaConf.from_dotlist(dotlist)

# add/merge to existing config tree
cfg = OmegaConf.merge(cfg, override)

print(cfg.strategy.fast_window, cfg.backtest.commission)   # 5 0.002

**3. Checking variable types against a schema**
<br> <br>

This is an extra feature included with OmegaConf that ensures hyperparameter variable types, and throws errors early if a type mismatch exists.

In [ ]:
# Creates an unstructured list of variable types for each hyperparameter
@dataclass
class StrategyConfig:
    name: str
    fast_window: int
    slow_window: int

schema = OmegaConf.structured(StrategyConfig)

# Merge is unsuccessful if schema and strategy config variable types disagree
checked = OmegaConf.merge(schema, strategy)                # passes
print(checked.slow_window)                                 # 30

# Break it on purpose, one line at a time (each raises, so run them separately):
OmegaConf.merge(schema, {"fast_window": "quick"})          # ValidationError: "quick" is not an int
OmegaConf.merge(schema, {"fast_windw": 5})                 # ConfigKeyError: typo caught

**4. Hydra: groups and CLI overrides** \
\
Omegaconf has a built in configuration management tool called Hydra. it takes the config tree from a default list, gives it to the funtion, and lets the command long override anything. The folder names are the config groups (market, strategy, and backtest).\
\
If you look in the default.yaml, you will see the primary market, strategy, and backtest configs. Each line means "take conf/'group'/'option'.yaml and put it under the key 'group'". That is why segment 2 wrapped each file in market, strategy and backtest by hand.\
\
Now we head over to the run.py file, where using 'python run.py' in the terminal will output the results of the run using the primary configs.\
\
Output: RESULT AAPL sma 10/30: total_return=+26.6%, max_drawdown=-9.6% | final $12,658.

**4.1.  Override from the CLI**\
\
Run each line and read the output before the next:

In [ ]:
'''
python run.py strategy.fast_window=5 backtest.commission=0.002   # change values
python run.py market=tsla                                        # swap a whole group option
python run.py +backtest.note=trial1                              # + adds a key that does not exist
python run.py strategy.fast_window=50                            # trips the fast < slow check
python run.py --cfg job                                          # print the config, do not run main()
'''

**5: Multirun sweep**\
\
The -m flag runs run.py once per comma-separated value, with no change to the code.\
\
1: Launching 3 jobs locally, then three result lines identical to the ones in the Overview\
\
2: Sweep over the fast_window parameter with a value of 5, 10, and 20\
\
3: Sweep over two parameters, market and fast_window, with 2 different options each. This produces 4 different outcomes.

In [ ]:
#1) python run.py -m market=aapl,msft,tsla
#2) python run.py -m strategy.fast_window=5,10,20
#3) python run.py -m market=aapl,tsla strategy.fast_window=5,20

**6. Custom resolvers: running Python inside a config**
<br> <br>
OmegaConf lets a config value call a Python function. You register the function once with `OmegaConf.register_new_resolver("name", function)`, and then any value written as `${name:arg1,arg2}` calls it. Arguments can be plain numbers or strings, or other config values: `${.fast_window}` means "fast_window in this same section".
<br> <br>
The function runs every time the value is read, not when the file is loaded, so a derived value stays correct after an override. ML and RL training configs use this for hyperparameters that depend on other hyperparameters, such as a slow window tied to a fast window, a learning rate scaled by batch size, or an exploration rate that decays as training goes on.

In [ ]:
# Register Python functions that configs can call by name.
# replace=True lets you re-run this cell without an "already registered" error.
OmegaConf.register_new_resolver("mul", lambda a, b: a * b, replace=True)
OmegaConf.register_new_resolver("daily_vol", lambda annual_vol: round(annual_vol / 252 ** 0.5, 4), replace=True)

derived = OmegaConf.create("""
market:
  annual_vol: 0.25
  daily_vol: ${daily_vol:${.annual_vol}}     # annual volatility -> per-day volatility
strategy:
  fast_window: 10
  slow_window: ${mul:${.fast_window},3}      # always 3x the fast window
""")

print(derived.market.daily_vol)       # 0.0157
print(derived.strategy.slow_window)   # 30

# Resolvers run when the value is read, so derived values follow overrides:
derived.strategy.fast_window = 5
print(derived.strategy.slow_window)   # 15

**6.1. A training schedule (RL example)**
<br> <br>
An epsilon-greedy RL agent picks a random action with probability epsilon. Epsilon starts high so the agent explores, then decays as training goes on so it exploits what it has learned. A resolver lets the config hold the whole schedule: the training loop only updates the step counter and reads `epsilon`.

In [ ]:
def linear_decay(start, end, step, total_steps):
    """Move from start to end over total_steps, then hold at end."""
    progress = min(step / total_steps, 1.0)
    return round(start + (end - start) * progress, 3)

OmegaConf.register_new_resolver("linear_decay", linear_decay, replace=True)

rl = OmegaConf.create({
    "train": {"step": 0, "total_steps": 10_000},
    "explore": {"epsilon": "${linear_decay:1.0,0.05,${train.step},${train.total_steps}}"},
})

for step in [0, 2_500, 5_000, 10_000, 15_000]:
    rl.train.step = step                       # what a training loop does each step
    print(f"step {step:>6}: epsilon = {rl.explore.epsilon}")

**6.2. Resolvers with Hydra**
<br> <br>
`run.py` registers the same `mul` resolver before Hydra loads the configs, so any YAML file in `conf/` can use it. The new `conf/strategy/sma_ratio.yaml` sets the slow window as a multiple of the fast window:

```yaml
name: sma_ratio
fast_window: 10
ratio: 3
slow_window: ${mul:${.fast_window},${.ratio}}
```

Overriding `fast_window` now moves `slow_window` along with it, and a multirun sweep keeps the 1:3 ratio in every job. The config that `run.py` prints still shows the `${mul:...}` text, because it is printed before anything is resolved. Add `--resolve` after `--cfg job` to see the computed values.

In [ ]:
#1) python run.py strategy=sma_ratio                                     # same result as the default: 10/30
#2) python run.py strategy=sma_ratio strategy.fast_window=5              # slow_window follows: 5/15
#3) python run.py strategy=sma_ratio strategy.fast_window=5 --cfg job --resolve   # print computed values
#4) python run.py -m strategy=sma_ratio strategy.fast_window=5,10,20     # sweep keeps the 1:3 ratio
#5) python run.py strategy=sma_ratio strategy.ratio=1                    # slow == fast trips the fast < slow check

**7: Q&A, troubleshooting**\
\
...
